[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/02_tres_pilares/02_proyecto_tres_pilares.ipynb)

# Proyecto 02 · Diez incidentes de SupportOps: ¿qué señal los detecta y por qué?

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo simulado; funciona en Colab) |
| **RAM aproximada** | < 400 MB |
| **Prerrequisitos** | Lección `02_tres_pilares` y proyecto 01 (fallos inyectados) |
| **Conexión** | Convierte las «huellas» del proyecto 01 en una **política de alertas**: qué señal vigilar para qué fallo. Es la hoja de ruta de los módulos 03-10 |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Tras tres meses en producción, el equipo de on-call os pide un **mapa de detección**: para cada tipo de incidente que habéis sufrido, ¿qué señal salta primero, cuál sirve para localizar y cuáles son ciegas?
Tienes **telemetría de 10 incidentes simulados** (cada uno: 30 min de línea base «normal» + 30 min con el incidente) en el formato de OpenTelemetry (ejecuciones, *spans* y *logs*). Tu trabajo: **medir** y **clasificar**.

## Datos

`simulate(nombre)` genera la telemetría de un incidente con semilla fija (determinista). Los incidentes: `llm_outage`, `llm_slow`, `tool_errors`, `tool_timeout`, `retry_storm`, `tool_loop`, `hallucination`, `pii_leak`, `cost_drift`, `traffic_spike`.
No hay datasets externos; **no se necesitan servicios** (no hay modo degradado porque nada depende del stack).

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, `mock`, semilla 42) |
|---|---|---|---|
| R1 | `per_minute_metrics` | Para `llm_outage`: 60 filas; `requests` suma 200; la tasa de error media por minuto de la fase B ≈ la tasa global medida por la solución | tolerancia ±0,05 |
| R2 | `culprit_name` | Sobre `llm_slow`, `tool_timeout` y `retry_storm` devuelve el mismo span responsable que la solución en ≥ 90 % de las trazas anómalas | ≥ 90 % |
| R3 | `phase_features` | Las 9 características coinciden con la solución (tolerancia relativa 2 %) en 3 incidentes de prueba | 3/3 |
| R4 | `fires` (qué señales saltan) | El conjunto de señales (`M`étricas, `L`ogs, `T`razas) coincide con la verdad en ≥ 9 de 10 incidentes | ≥ 9/10 |
| R5 | `diagnose` | Etiqueta correcta (incluye `sin_senal`) en ≥ 8 de 10 incidentes | ≥ 8/10 |
| R6 | Puntos ciegos | Identificas **exactamente** los incidentes donde *ninguna* señal salta y los que **no ve la métrica** pero sí logs/trazas | coincidencia exacta |

> 📏 Los umbrales salen de la solución de referencia, comprobada con 5 semillas distintas (42, 7, 99, 123, 2024): las clasificaciones son estables. No dependen de tu máquina (el tiempo es simulado).

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import json, re, collections
_ = check_stack([])      # este proyecto no necesita servicios

## Paso 0 · El simulador de incidentes (te lo damos hecho)

Cada incidente tiene **dos fases de 30 min**: **A = línea base** (sin fallos) y **B = incidente**. Se registra con `RecordingInstrumentor`, así que tienes `runs` (una fila por ticket), `spans` y `logs` con el esquema del curso.
`cost_drift` simula que alguien cambió el modelo por uno 12× más caro en la fase B; `traffic_spike` triplica el tráfico en B.

In [ ]:
A_START, B_START = SIM_EPOCH_S, SIM_EPOCH_S + 1800
SCENARIOS = {   # nombre: (tipo de fallo en la fase B, probabilidad por ticket, parámetros)
    "llm_outage":    ("llm_error", 0.6, {"status": 503, "persistent": True}),
    "llm_slow":      ("latency_spike", 0.8, {"factor": 6.0, "target": "llm"}),
    "tool_errors":   ("tool_error", 0.5, {}),
    "tool_timeout":  ("tool_timeout", 0.4, {"timeout_s": 5.0}),
    "retry_storm":   ("llm_error", 0.6, {"status": 429, "persistent": False}),
    "tool_loop":     ("tool_loop", 0.4, {}),
    "hallucination": ("hallucination", 0.5, {}),
    "pii_leak":      ("pii_leak", 0.3, {}),
    "cost_drift":    (None, 0, {}),
    "traffic_spike": (None, 0, {}),
}

def simulate(name: str, seed: int = SEED, n_base: int = 100) -> dict:
    kind, prob, params = SCENARIOS[name]
    n_b = n_base * (3 if name == "traffic_spike" else 1)
    ta = generate_tickets(n_base, seed, duration_s=1800, start_s=A_START)
    inc = [Incident(name, kind, 0, 1800, prob, params)] if kind else []
    tb = generate_tickets(n_b, seed + 1, duration_s=1800, start_s=B_START, incidents=inc)
    rec = RecordingInstrumentor()
    small = SupportAgent(llm=MockLLM("mock-small", seed=seed), instrumentor=rec)
    big = SupportAgent(llm=MockLLM("mock-large" if name == "cost_drift" else "mock-small", seed=seed), instrumentor=rec)
    rows = to_rows(ta, run_tickets(small, ta)) + to_rows(tb, run_tickets(big, tb))
    runs = pd.DataFrame(rows)
    runs["phase"] = np.where(runs.created_at < B_START, "A", "B"); runs["minute"] = ((runs.created_at - A_START) // 60).astype(int)
    runs["steps"] = runs.n_llm_calls + runs.n_tool_calls
    spans = pd.DataFrame(rec.spans); spans["dur"] = (spans.end_unix_nano - spans.start_unix_nano) / 1e9
    logs = pd.DataFrame(rec.logs)
    return {"name": name, "runs": runs, "spans": spans, "logs": logs}

INC = {n: simulate(n) for n in SCENARIOS}
print({n: (len(d["runs"]), len(d["spans"]), len(d["logs"])) for n, d in INC.items()})
INC["llm_outage"]["runs"].head(3)[["ticket_id", "phase", "minute", "status", "latency_s", "steps", "input_tokens", "cost_usd"]]

## Paso 1 · La señal de **métricas**: tabla por minuto

Las métricas son agregados. **TODO 1.** Implementa `per_minute_metrics(runs)` que devuelva un `DataFrame` con **60 filas** (índice `minute` de 0 a 59, rellena con 0/NaN los minutos vacíos) y columnas:
`phase` (A/B), `requests`, `error_rate` (fracción de `status` en {`error`,`max_steps`}), `p95_latency_s`, `tokens_in_per_req`, `cost_per_req`, `steps_per_req`.

<details>
<summary>🪜 Pista 1</summary>

`runs.groupby('minute')` con `.agg(...)`, y después `.reindex(range(60))`. Para `error_rate` usa `status.isin([...]).mean()`; para el p95 `quantile(0.95)`. La fase de un minuto es `'A'` si `minute < 30`, `'B'` si no.

</details>

<details>
<summary>🪜 Pista 2</summary>

Estructura: `g = runs.groupby('minute'); out = pd.DataFrame({'requests': g.size(), 'error_rate': g.status.apply(lambda s: s.isin(['error','max_steps']).mean()), 'p95_latency_s': g.latency_s.quantile(.95), 'tokens_in_per_req': g.input_tokens.mean(), 'cost_per_req': g.cost_usd.mean(), 'steps_per_req': g.steps.mean()}).reindex(range(60))` y rellena `requests` con 0 y `phase` con la regla de los 30 min.

</details>

In [ ]:
def per_minute_metrics(runs: pd.DataFrame) -> pd.DataFrame:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    PM = per_minute_metrics(INC["llm_outage"]["runs"]); display(PM.iloc[[0, 1, 29, 30, 45, 59]].round(3))
except NotImplementedError as e:
    PM = None; print("Pendiente:", e)

## Paso 2 · Características por fase (métricas + logs + trazas)

Para decidir si una señal «salta», compara la fase **B** (incidente) con la **A** (base). Calcularás 9 características. Las de métricas y logs son agregaciones; la de trazas requiere **mirar dentro de los árboles de spans**.

| Característica | Señal | Definición (B frente a A) |
|---|---|---|
| `err_delta` | métricas | `error_rate`(B) − `error_rate`(A) sobre `runs` (error = `error` o `max_steps`) |
| `p95_ratio` | métricas | p95 de `latency_s`(B) / p95(A) |
| `tokens_ratio` | métricas | media de `input_tokens`(B) / media(A) |
| `cost_ratio` | métricas | media de `cost_usd`(B) / media(A) |
| `steps_ratio` | métricas | media de `steps`(B) / media(A) |
| `traffic_ratio` | métricas | nº de tickets(B) / nº de tickets(A) |
| `warn_per_req_delta` | logs | (logs con `severity_number` ≥ 13 por ticket)(B) − (A). Para atribuir un log a una fase, une `logs.trace_id` con la traza del ticket |
| `child_err_delta` | trazas | fracción de trazas con **algún span hijo en ERROR**(B) − (A) |
| `repeat_share_delta` | trazas | fracción de trazas con **≥ 3 spans `execute_tool` con el mismo nombre**(B) − (A) |

Además necesitas `slow_share` (fracción de trazas de B «anómalas»: con un hijo en ERROR **o** duración de la raíz > 2 × p95 de A) y `purity` (de las anómalas, fracción cuyo `culprit_name` es el más frecuente).

**TODO 2.** `culprit_name(trace_spans)`: dado el `DataFrame` de los spans de **una** traza, devuelve el `name` del span **hijo** responsable: el hijo en ERROR de **mayor duración** si hay alguno; si no, el hijo de mayor duración. (Si no hay hijos, `None`.)

<details>
<summary>🪜 Pista 1</summary>

Filtra los hijos con `trace_spans[trace_spans.parent_span_id.notna()]`. Si hay `status == 'ERROR'`, quédate con esos; ordena por `dur` y toma el último (`.iloc[-1]['name']`).

</details>

In [ ]:
def culprit_name(trace_spans: pd.DataFrame):
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    d = INC["tool_timeout"]; tr = d["spans"][d["spans"].trace_id == d["spans"][d["spans"].status == "ERROR"].trace_id.iloc[0]]
    print(culprit_name(tr))
except NotImplementedError as e:
    print("Pendiente:", e)

**TODO 3.** `phase_features(inc)` devuelve un `dict` con las 9 características + `slow_share`, `purity` y `culprit` (el nombre más frecuente, o `None`). Te damos como ayuda el mapa `trace → fase` y los tiempos de A.

<details>
<summary>🪜 Pista 1</summary>

Mapas útiles: `t2p` (trace_id → fase) se obtiene uniendo los spans raíz con `runs` por el atributo `supportops.ticket.id`. Para logs: `logs['phase'] = logs.trace_id.map(t2p)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Por traza: `has_err_child = ((g.status == 'ERROR') & g.parent_span_id.notna()).any()`; `repeat = (g[g.name.str.startswith('execute_tool')].name.value_counts() >= 3).any()`; duración de la raíz = `g[g.parent_span_id.isna()].dur.iloc[0]`. Agrupa con `spans.groupby('trace_id')`.

</details>

In [ ]:
def phase_features(inc: dict) -> dict:
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    FEATS = {n: phase_features(d) for n, d in INC.items()}
    display(pd.DataFrame(FEATS).T.round(2))
except NotImplementedError as e:
    FEATS = None; print("Pendiente:", e)

## Paso 3 · ¿Qué señal salta?

**TODO 4.** `fires(f)` devuelve el conjunto de señales que **saltan**: `"M"` (métricas), `"L"` (logs), `"T"` (trazas). Reglas del equipo (las mismas en la solución):

- **M** si: `err_delta ≥ 0,10` **o** `p95_ratio ≥ 2` **o** `steps_ratio ≥ 1,4` **o** `steps_ratio ≤ 0,75` **o** `cost_ratio ≥ 2` **o** `traffic_ratio ≥ 2` **o** `tokens_ratio ≥ 1,5` **o** `tokens_ratio ≤ 0,6`.
- **L** si `warn_per_req_delta ≥ 0,15`.
- **T** si `child_err_delta ≥ 0,10` **o** `repeat_share_delta ≥ 0,10` **o** (`slow_share ≥ 0,25` **y** `purity ≥ 0,7`).

*(Reflexiona: ¿por qué las reglas de M incluyen umbrales «≤»? ¿Qué incidente los necesita?)*

In [ ]:
def fires(f: dict) -> set:
    # TODO 4
    raise NotImplementedError("TODO 4")

try:
    FIRED = {n: fires(f) for n, f in FEATS.items()}
    display(pd.DataFrame({n: {"M": "M" in s, "L": "L" in s, "T": "T" in s} for n, s in FIRED.items()}).T.replace({True: "✔", False: "·"}))
except (NotImplementedError, TypeError, AttributeError) as e:
    FIRED = None; print("Pendiente:", type(e).__name__)

## Paso 4 · Diagnóstico

**TODO 5.** `diagnose(f, fired)` devuelve **una** etiqueta de entre
`llm_outage, llm_slow, tool_errors, tool_timeout, retry_storm, tool_loop, hallucination, cost_drift, traffic_spike, sin_senal`.
Usa las características (y `f["culprit"]`, que dice si el responsable es un span `chat…` o `execute_tool…`). `sin_senal` significa «ninguna señal salta: no hay nada que diagnosticar con los tres pilares».
Piensa el **orden** de las reglas: de lo más específico (tráfico, coste) a lo más general.

<details>
<summary>🪜 Pista 1</summary>

Un posible árbol: (1) `traffic_ratio ≥ 2` → `traffic_spike`; (2) `cost_ratio ≥ 2` y `tokens_ratio < 1.5` y `steps_ratio` cercano a 1 → `cost_drift`; (3) nada salta (`fired` vacío) → `sin_senal`; (4) `repeat_share_delta ≥ 0.1` (o pasos y tokens muy altos) → `tool_loop`; (5) errores en el span `chat` con `err_delta ≥ 0.1` → `llm_outage`…

</details>

<details>
<summary>🪜 Pista 2</summary>

Continuación: (6) hijos en ERROR y `err_delta < 0.1` (la raíz sale «ok»): si el culpable empieza por `chat` → `retry_storm`; si por `execute_tool` → `tool_errors`, salvo que además `p95_ratio ≥ 2` → `tool_timeout`; (7) `p95_ratio ≥ 2` con culpable `chat` → `llm_slow`; (8) solo M por `steps_ratio ≤ 0.75` → `hallucination`.

</details>

In [ ]:
LABELS = ["llm_outage", "llm_slow", "tool_errors", "tool_timeout", "retry_storm", "tool_loop", "hallucination", "cost_drift", "traffic_spike", "sin_senal"]

def diagnose(f: dict, fired: set) -> str:
    # TODO 5
    raise NotImplementedError("TODO 5")

try:
    DIAG = {n: diagnose(FEATS[n], FIRED[n]) for n in INC}
    display(pd.DataFrame({"diagnóstico": DIAG, "esperado": {n: n if n != "pii_leak" else "sin_senal" for n in INC}}))
except (NotImplementedError, TypeError) as e:
    DIAG = None; print("Pendiente:", type(e).__name__)

## Paso 5 · Informe final y puntos ciegos

**TODO 6.** Rellena `REPORT`: para cada incidente, un `dict` con `fired` (el conjunto), `blind` (las señales de {M, L, T} que **no** saltan), `first_alert` (la señal con la que **alertarías** primero: prioriza M si salta —las métricas son baratas y rápidas—, si no L, si no T, si no `None`) y `evidence` (una frase con **al menos un número** de tus características que respalde el diagnóstico).
Después contesta en `BLIND_SPOTS` dos listas: los incidentes donde **ninguna** señal salta y los incidentes que **la métrica no ve** pero sí logs o trazas.

In [ ]:
REPORT = {}
BLIND_SPOTS = {"ninguna_senal": [], "invisibles_para_metricas": []}

def build_report():
    # TODO 6: rellena REPORT[nombre] = {"fired": set, "blind": set, "first_alert": "M"|"L"|"T"|None, "evidence": str} y BLIND_SPOTS
    raise NotImplementedError("TODO 6")

try:
    build_report()
    display(pd.DataFrame(REPORT).T[["fired", "blind", "first_alert"]]); print(BLIND_SPOTS)
except (NotImplementedError, TypeError, KeyError) as e:
    print("Pendiente:", type(e).__name__)

## Paso 6 · Rúbrica

In [ ]:
# Valores de verdad (calibrados con la solución de referencia sobre varias semillas). Van codificados en base64 SOLO para no
# estropearte el ejercicio: no los decodifiques antes de intentarlo (la rúbrica los usa por ti).
import base64
_T = json.loads(base64.b64decode("eyJmaXJlZCI6eyJsbG1fb3V0YWdlIjoiTUxUIiwibGxtX3Nsb3ciOiJNVCIsInRvb2xfZXJyb3JzIjoiTFQiLCJ0b29sX3RpbWVvdXQiOiJNTFQiLCJyZXRyeV9zdG9ybSI6IkxUIiwidG9vbF9sb29wIjoiTUxUIiwiaGFsbHVjaW5hdGlvbiI6Ik0iLCJwaWlfbGVhayI6IiIsImNvc3RfZHJpZnQiOiJNIiwidHJhZmZpY19zcGlrZSI6Ik0ifSwiYmxpbmQiOnsibmluZ3VuYV9zZW5hbCI6WyJwaWlfbGVhayJdLCJpbnZpc2libGVzX3BhcmFfbWV0cmljYXMiOlsicmV0cnlfc3Rvcm0iLCJ0b29sX2Vycm9ycyJdfSwicmVmIjp7ImxsbV9vdXRhZ2UiOnsiZXJyX2RlbHRhIjowLjU2LCJwOTVfcmF0aW8iOjAuOTEzOCwidG9rZW5zX3JhdGlvIjowLjQ0NjQsImNvc3RfcmF0aW8iOjAuNDUxOSwic3RlcHNfcmF0aW8iOjAuODAyNSwidHJhZmZpY19yYXRpbyI6MS4wLCJ3YXJuX3Blcl9yZXFfZGVsdGEiOjEuNjgsImNoaWxkX2Vycl9kZWx0YSI6MC41NiwicmVwZWF0X3NoYXJlX2RlbHRhIjowLjB9LCJ0b29sX2Vycm9ycyI6eyJlcnJfZGVsdGEiOjAuMCwicDk1X3JhdGlvIjowLjg0NTIsInRva2Vuc19yYXRpbyI6MC45MDAxLCJjb3N0X3JhdGlvIjowLjg4OSwic3RlcHNfcmF0aW8iOjAuOTQyNywidHJhZmZpY19yYXRpbyI6MS4wLCJ3YXJuX3Blcl9yZXFfZGVsdGEiOjAuOTIsImNoaWxkX2Vycl9kZWx0YSI6MC40OCwicmVwZWF0X3NoYXJlX2RlbHRhIjowLjB9LCJ0b29sX2xvb3AiOnsiZXJyX2RlbHRhIjowLjQxLCJwOTVfcmF0aW8iOjIuNzQzNSwidG9rZW5zX3JhdGlvIjoyLjQ1MDMsImNvc3RfcmF0aW8iOjIuMzAzLCJzdGVwc19yYXRpbyI6Mi4xMTc4LCJ0cmFmZmljX3JhdGlvIjoxLjAsIndhcm5fcGVyX3JlcV9kZWx0YSI6MC40MSwiY2hpbGRfZXJyX2RlbHRhIjowLjAsInJlcGVhdF9zaGFyZV9kZWx0YSI6MC40MX19fQ=="))
TRUTH_FIRED, REF_FEATS = _T["fired"], _T["ref"]
TRUTH_DIAG = {n: n for n in INC} | {"pii_leak": "sin_senal"}

def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    # R1
    try:
        pm = per_minute_metrics(INC["llm_outage"]["runs"]); b = pm[pm.phase == "B"]
        ref = INC["llm_outage"]["runs"].query("phase == 'B'").status.isin(["error", "max_steps"]).mean()
        ok = len(pm) == 60 and int(pm.requests.sum()) == 200 and abs(b.error_rate.mean() - ref) < 0.05
        add("R1", "per_minute_metrics (60 filas, 200 peticiones, tasa de error coherente)", ok, f"filas={len(pm)} peticiones={int(pm.requests.sum())} err(B)≈{b.error_rate.mean():.2f} vs {ref:.2f}")
    except Exception as e: add("R1", "per_minute_metrics", False, type(e).__name__)
    # R2
    try:
        hits = tot = 0
        for n in ("llm_slow", "tool_timeout", "retry_storm"):
            sp = INC[n]["spans"]; B = set(INC[n]["runs"].query("phase == 'B'").ticket_id)
            roots = sp[sp.parent_span_id.isna()]; tb = set(roots[roots.attributes.map(lambda a: a["supportops.ticket.id"]).isin(B)].trace_id)
            for t, g in sp[sp.trace_id.isin(tb)].groupby("trace_id"):
                ch = g[g.parent_span_id.notna()]; ref_ch = ch[ch.status == "ERROR"] if (ch.status == "ERROR").any() else ch
                if len(ch) == 0: continue
                tot += 1; hits += culprit_name(g) == ref_ch.sort_values("dur").iloc[-1]["name"]
        add("R2", "culprit_name coincide con la referencia (≥ 90 %)", tot > 0 and hits / tot >= 0.9, f"{hits}/{tot}")
    except Exception as e: add("R2", "culprit_name", False, type(e).__name__)
    # R3
    try:
        ref_cols = ["err_delta", "p95_ratio", "tokens_ratio", "cost_ratio", "steps_ratio", "traffic_ratio", "warn_per_req_delta", "child_err_delta", "repeat_share_delta"]
        good = 0
        for n in ("llm_outage", "tool_errors", "tool_loop"):
            f, g = phase_features(INC[n]), REF_FEATS[n]
            good += all(abs(f[c] - g[c]) <= 0.02 * max(abs(g[c]), 1e-9) + 1e-6 for c in ref_cols)
        add("R3", "phase_features coincide con la referencia (3 incidentes)", good == 3, f"{good}/3")
    except Exception as e: add("R3", "phase_features", False, type(e).__name__)
    # R4
    try:
        n_ok = sum(set(TRUTH_FIRED[n]) == fires(FEATS[n]) for n in INC)
        add("R4", "fires: señales correctas en ≥ 9/10", n_ok >= 9, f"{n_ok}/10")
    except Exception as e: add("R4", "fires", False, type(e).__name__)
    # R5
    try:
        n_ok = sum(diagnose(FEATS[n], fires(FEATS[n])) == TRUTH_DIAG[n] for n in INC)
        add("R5", "diagnose: etiqueta correcta en ≥ 8/10", n_ok >= 8, f"{n_ok}/10")
    except Exception as e: add("R5", "diagnose", False, type(e).__name__)
    # R6
    try:
        exp = _T["blind"]
        got = {k: sorted(v) for k, v in BLIND_SPOTS.items()}
        ev_ok = all(isinstance(REPORT[n]["evidence"], str) and re.search(r"\d", REPORT[n]["evidence"]) for n in INC)
        add("R6", "puntos ciegos exactos + evidencias con números", got == {k: sorted(v) for k, v in exp.items()} and ev_ok, f"{got}")
    except Exception as e: add("R6", "puntos ciegos", False, type(e).__name__)
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Retos extra (nivel experto)

1. **Robustez:** repite el análisis con las semillas 7, 99 y 123 (`simulate(n, seed=...)`). ¿Qué reglas de `fires` están más cerca de sus umbrales? ¿Qué incidente es más frágil y por qué (pista: `cost_drift` en trazas)?
2. **Tiempo hasta la detección:** en vez de comparar fases enteras, calcula con ventana deslizante de 3 min el **primer minuto** en que cada regla de M salta tras el inicio del incidente. ¿Qué incidentes detectan las métricas en < 3 min y cuáles tardan más? ¿Cómo cambia con 3× menos tráfico?
3. **El «cuarto pilar»:** añade a `diagnose` una señal de **contenido** (por ejemplo, el regex de PII del proyecto 01 sobre `runs.answer`). ¿Cuántos incidentes pasan de `sin_senal` a diagnosticados? ¿A qué coste (falsos positivos, privacidad)?
4. **Cardinalidad:** calcula cuántas series generaría un histograma por `customer_id` en estos incidentes (`runs.customer_id.nunique()`) y diseña una alternativa con trazas para atribuir latencia por cliente.
5. **Alerta multi-señal:** propón una regla de alerta que combine `error_rate` y `p95` y estima su tasa de falsos positivos en las fases A de los 10 incidentes.

## Reflexión (producción e hilo conductor)

- `tool_errors` y `retry_storm` son **invisibles para las métricas de la raíz**. ¿Qué métrica *adicional* (con qué etiquetas acotadas) las haría visibles sin disparar la cardinalidad?
- `hallucination` solo se veía por una **caída** de pasos/tokens. ¿Qué pasaría si el agente alucinara *y además* hiciera llamadas innecesarias a herramientas (pasos iguales)? ¿Qué señal sobreviviría?
- Tu `first_alert` favorece a M por ser barata. ¿En qué incidente esa preferencia te llevaría a una alerta **ruidosa o tardía** y qué señal mejoraría el *tiempo de diagnóstico*?
- En el módulo 03 instrumentarás SupportOps con el SDK: ¿qué atributos pondrías para que `culprit_name` (y el análisis por herramienta/modelo) funcionase sin tocar el código de análisis?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
ERR = ["error", "max_steps"]

# TODO 1
def per_minute_metrics(runs: pd.DataFrame) -> pd.DataFrame:
    g = runs.groupby("minute")
    out = pd.DataFrame({"requests": g.size(), "error_rate": g.status.apply(lambda s: s.isin(ERR).mean()), "p95_latency_s": g.latency_s.quantile(.95),
                        "tokens_in_per_req": g.input_tokens.mean(), "cost_per_req": g.cost_usd.mean(), "steps_per_req": g.steps.mean()}).reindex(range(60))
    out["requests"] = out.requests.fillna(0).astype(int)
    out.insert(0, "phase", ["A" if m < 30 else "B" for m in range(60)])
    out.index.name = "minute"
    return out

# TODO 2
def culprit_name(trace_spans: pd.DataFrame):
    ch = trace_spans[trace_spans.parent_span_id.notna()]
    if len(ch) == 0: return None
    err = ch[ch.status == "ERROR"]
    return (err if len(err) else ch).sort_values("dur").iloc[-1]["name"]

# TODO 3
def phase_features(inc: dict) -> dict:
    runs, spans, logs = inc["runs"], inc["spans"], inc["logs"]
    A, B = runs[runs.phase == "A"], runs[runs.phase == "B"]
    f = {}
    err = lambda d: d.status.isin(ERR).mean()
    f["err_delta"] = err(B) - err(A)
    f["p95_ratio"] = B.latency_s.quantile(.95) / A.latency_s.quantile(.95)
    f["tokens_ratio"] = B.input_tokens.mean() / A.input_tokens.mean()
    f["cost_ratio"] = B.cost_usd.mean() / A.cost_usd.mean()
    f["steps_ratio"] = B.steps.mean() / A.steps.mean()
    f["traffic_ratio"] = len(B) / len(A)
    roots = spans[spans.parent_span_id.isna()].copy()
    roots["ticket"] = roots.attributes.map(lambda a: a["supportops.ticket.id"])
    t2p = dict(zip(roots.trace_id, roots.ticket.map(dict(zip(runs.ticket_id, runs.phase)))))
    w = logs[logs.severity_number >= 13].copy(); w["phase"] = w.trace_id.map(t2p)
    f["warn_per_req_delta"] = (w.phase == "B").sum() / len(B) - (w.phase == "A").sum() / len(A)
    by = {t: g for t, g in spans.groupby("trace_id")}
    has_err = lambda g: bool(((g.status == "ERROR") & g.parent_span_id.notna()).any())
    repeat = lambda g: bool((g[g.name.str.startswith("execute_tool")].name.value_counts() >= 3).any())
    ids = {p: [t for t, q in t2p.items() if q == p] for p in "AB"}
    share = lambda fn, p: float(np.mean([fn(by[t]) for t in ids[p]]))
    f["child_err_delta"] = share(has_err, "B") - share(has_err, "A")
    f["repeat_share_delta"] = share(repeat, "B") - share(repeat, "A")
    p95a = A.latency_s.quantile(.95)
    anom = [t for t in ids["B"] if has_err(by[t]) or by[t][by[t].parent_span_id.isna()].dur.iloc[0] > 2 * p95a]
    f["slow_share"] = len(anom) / max(1, len(ids["B"]))
    c = collections.Counter(culprit_name(by[t]) for t in anom)
    f["purity"] = c.most_common(1)[0][1] / len(anom) if anom else 0.0
    f["culprit"] = c.most_common(1)[0][0] if anom else None
    return f

# TODO 4
def fires(f: dict) -> set:
    s = set()
    if (f["err_delta"] >= .10 or f["p95_ratio"] >= 2 or f["steps_ratio"] >= 1.4 or f["steps_ratio"] <= .75 or f["cost_ratio"] >= 2
            or f["traffic_ratio"] >= 2 or f["tokens_ratio"] >= 1.5 or f["tokens_ratio"] <= .6): s.add("M")
    if f["warn_per_req_delta"] >= .15: s.add("L")
    if f["child_err_delta"] >= .10 or f["repeat_share_delta"] >= .10 or (f["slow_share"] >= .25 and f["purity"] >= .7): s.add("T")
    return s

# TODO 5
def diagnose(f: dict, fired: set) -> str:
    cul = (f.get("culprit") or "")
    if f["traffic_ratio"] >= 2: return "traffic_spike"
    if f["cost_ratio"] >= 2 and f["tokens_ratio"] < 1.5 and .8 <= f["steps_ratio"] <= 1.25: return "cost_drift"
    if not fired: return "sin_senal"
    if f["repeat_share_delta"] >= .10 or (f["steps_ratio"] >= 1.4 and f["tokens_ratio"] >= 1.5): return "tool_loop"
    if f["err_delta"] >= .10 and cul.startswith("chat"): return "llm_outage"
    if f["child_err_delta"] >= .10:                                  # la raíz sale «ok»: el fallo vive dentro
        if cul.startswith("chat"): return "retry_storm"
        return "tool_timeout" if f["p95_ratio"] >= 2 else "tool_errors"
    if f["p95_ratio"] >= 2 and cul.startswith("chat"): return "llm_slow"
    if f["steps_ratio"] <= .75: return "hallucination"
    return "sin_senal"

# TODO 6
def build_report():
    for n in INC:
        f, s = FEATS[n], FIRED[n]
        REPORT[n] = {"fired": s, "blind": set("MLT") - s, "first_alert": next((x for x in "MLT" if x in s), None),
                     "evidence": f"err_delta={f['err_delta']:.2f}, p95×{f['p95_ratio']:.1f}, pasos×{f['steps_ratio']:.2f}, tokens×{f['tokens_ratio']:.2f}, coste×{f['cost_ratio']:.1f}, tráfico×{f['traffic_ratio']:.1f}"}
    BLIND_SPOTS["ninguna_senal"] = [n for n in INC if not FIRED[n]]
    BLIND_SPOTS["invisibles_para_metricas"] = [n for n in INC if "M" not in FIRED[n] and FIRED[n]]

FEATS = {n: phase_features(d) for n, d in INC.items()}
FIRED = {n: fires(f) for n, f in FEATS.items()}
DIAG = {n: diagnose(FEATS[n], FIRED[n]) for n in INC}
build_report()
display(pd.DataFrame(FEATS).T.round(2))
display(pd.DataFrame({"fired": {n: "".join(sorted(s, key="MLT".index)) or "—" for n, s in FIRED.items()}, "diagnóstico": DIAG, "primera alerta": {n: REPORT[n]["first_alert"] for n in INC}}))
print(BLIND_SPOTS)
display(run_rubric())

In [ ]:
# Visualización de la solución: ¿qué señal salta en cada incidente? (matriz) + series por minuto de tres incidentes representativos
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1.5]})
mat = np.array([[("M" in FIRED[n]), ("L" in FIRED[n]), ("T" in FIRED[n])] for n in INC], dtype=int)
axes[0].imshow(mat, cmap="Greens", vmin=0, vmax=1.4, aspect="auto"); axes[0].set_xticks(range(3)); axes[0].set_xticklabels(["métricas", "logs", "trazas"]); axes[0].set_yticks(range(len(INC))); axes[0].set_yticklabels(list(INC))
for i in range(mat.shape[0]):
    for j in range(3): axes[0].text(j, i, "✔" if mat[i, j] else "·", ha="center", va="center", fontsize=12)
axes[0].set_title("¿Qué señal salta? (medido)"); axes[0].grid(False)
for n, c in [("tool_errors", viz.PALETTE["warn"]), ("llm_slow", viz.PALETTE["error"]), ("traffic_spike", viz.PALETTE["traces"])]:
    pm = per_minute_metrics(INC[n]["runs"]); axes[1].plot(pm.index, pm.requests.rolling(3, min_periods=1).mean() / pm.requests[:30].mean(), color=c, ls=":", lw=1.2)
    axes[1].plot(pm.index, (pm.p95_latency_s / pm.p95_latency_s[:30].mean()).rolling(3, min_periods=1).mean(), color=c, lw=2, label=f"{n}: p95 relativo (—) / tráfico (···)")
axes[1].axvline(30, color="k", ls="--"); axes[1].set_xlabel("minuto"); axes[1].set_ylabel("relativo a la media de la fase A"); axes[1].legend(fontsize=8); axes[1].set_title("Lo que ve un dashboard de RED en tres incidentes (inicio del incidente: línea discontinua)")
plt.tight_layout(); plt.show()